# LAB 02 — Chunking desde cero

**Objetivo:** implementar chunking, configurar `chunk_size` y `overlap`, visualizar y comparar resultados.

**Tiempo sugerido:** 40 minutos · **No requiere API key**

Pasos: 1) leer el documento · 2) dividirlo · 3) configurar chunk size · 4) configurar overlap · 5) visualizar · 6) comparar.

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

## 1. Chunking por caracteres con overlap

```
CHUNK 1  [A B C D E]
CHUNK 2        [D E F G H]      ← overlap = D E
avance = chunk_size − overlap
```

### ✏️ Ejercicio 1 — `chunk_text`

- Si `overlap >= chunk_size` lanza `ValueError` (¿por qué?).
- Cada chunk empieza `chunk_size - overlap` caracteres después del anterior.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    # TODO
    ...

In [ ]:
# ✅ Autoverificación
assert chunk_text("ABCDEFGHIJ", 4, 0) == ["ABCD", "EFGH", "IJ"]
assert chunk_text("ABCDEFGHIJ", 4, 2) == ["ABCD", "CDEF", "EFGH", "GHIJ", "IJ"]
try:
    chunk_text("ABC", 3, 3)
    raise AssertionError("Debió lanzar ValueError")
except ValueError:
    pass
print("✅ Ejercicio 1 correcto")

In [ ]:
# Visualiza el overlap entre dos chunks consecutivos
c = chunk_text(texto, 300, 80)
print("CHUNK 0:\n", c[0], "\n")
print("CHUNK 1:\n", c[1], "\n")
print("TEXTO COMPARTIDO (overlap):\n", c[0][-80:])

## 2. Estimación de chunks

**chunks ≈ longitud / (chunk_size − overlap)**

### ✏️ Ejercicio 2 — `estimar_chunks`

Estima y compara con el número real para tu documento.

In [ ]:
import math

def estimar_chunks(longitud, chunk_size, overlap):
    # TODO
    ...

print("Estimado:", estimar_chunks(len(texto), 1000, 200), "| Real:", len(chunk_text(texto, 1000, 200)))

In [ ]:
# ✅ Autoverificación (ejemplo de la diapositiva: 10,000 / 800 ≈ 12.5 → 13)
assert estimar_chunks(10_000, 1000, 200) == 13
print("✅ Ejercicio 2 correcto")

## 3. Experimento: chunk pequeño, mediano y grande

### ✏️ Ejercicio 3 — tabla comparativa

Para cada combinación de `chunk_size` ∈ {150, 600, 2000} y `overlap` ∈ {0, 10 %, 30 %} calcula:
`n_chunks`, `promedio_caracteres` y `palabras_cortadas` (chunks cuyo final corta una palabra a la mitad — usa la función dada).

In [ ]:
import pandas as pd

def corta_palabra(chunk, siguiente):
    """True si el chunk termina a mitad de palabra."""
    return bool(chunk) and bool(siguiente) and chunk[-1].isalnum() and siguiente[0].isalnum()

def palabras_cortadas(text, chunk_size, overlap):
    paso = chunk_size - overlap
    cortes = 0
    for start in range(0, len(text), paso):
        fin = start + chunk_size
        if fin < len(text) and corta_palabra(text[fin - 1], text[fin]):
            cortes += 1
    return cortes

In [ ]:
filas = []
for size in [150, 600, 2000]:
    for pct in [0, 0.10, 0.30]:
        overlap = int(size * pct)
        # TODO: calcula los chunks y agrega una fila con size, overlap, n_chunks, promedio_caracteres, palabras_cortadas
        ...

experimento = pd.DataFrame(filas)
experimento

In [ ]:
# ✅ Autoverificación
assert len(experimento) == 9 and "n_chunks" in experimento.columns
print("✅ Ejercicio 3 correcto")

**Observa:** ¿cómo crece el número de chunks (y el almacenamiento) al subir el overlap?
¿Cuántas palabras se cortan con cualquier tamaño? → el problema no es el tamaño, es **cortar sin respetar la estructura**.

## 4. Chunking por estructura

### ✏️ Ejercicio 4 — `chunk_by_paragraph`

Agrupa párrafos (separados por línea en blanco `\n\n`) en chunks de hasta `max_chars` **sin partir ningún párrafo**.
Si un párrafo por sí solo excede `max_chars`, va solo en su propio chunk.

In [ ]:
def chunk_by_paragraph(text, max_chars=800):
    parrafos = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, actual = [], ""
    for p in parrafos:
        # TODO: si agregar p excede max_chars, guarda `actual` y empieza uno nuevo
        ...
    # TODO: no olvides el último chunk
    return chunks

In [ ]:
# ✅ Autoverificación
demo = "Uno uno.\n\nDos dos dos.\n\nTres."
assert chunk_by_paragraph(demo, 100) == ["Uno uno.\n\nDos dos dos.\n\nTres."]
assert chunk_by_paragraph(demo, 15) == ["Uno uno.", "Dos dos dos.", "Tres."]
assert "".join(chunk_by_paragraph(texto, 800)).replace("\n", "") == "".join(
    p.strip() for p in texto.split("\n\n") if p.strip()).replace("\n", "")
print("✅ Ejercicio 4 correcto")

### ✏️ Ejercicio 5 — `chunk_by_section` con metadata

Divide por encabezados Markdown (`#`, `##`, `###`...) y devuelve una lista de diccionarios:
`{"seccion": <título>, "texto": <título + contenido>}`. Incluir el título dentro del texto da contexto al chunk.

In [ ]:
import re

def chunk_by_section(text):
    resultado, seccion, buffer = [], "Inicio", []
    for linea in text.splitlines():
        m = re.match(r"^#{1,6}\s+(.*)", linea)
        if m:
            # TODO: guarda la sección anterior (si tiene contenido) y empieza una nueva
            ...
        else:
            buffer.append(linea)
    # TODO: guarda la última sección
    return resultado

In [ ]:
# ✅ Autoverificación
demo = "# Título\nintro\n## A\ntexto a\n## B\ntexto b"
r = chunk_by_section(demo)
assert [c["seccion"] for c in r] == ["Título", "A", "B"]
assert r[1]["texto"] == "A\ntexto a"
print("✅ Ejercicio 5 correcto")

secciones = chunk_by_section(texto)
print(f"{len(secciones)} chunks por sección")
pd.DataFrame([{"seccion": c["seccion"], "caracteres": len(c["texto"])} for c in secciones])

## 6. Compara las estrategias

Busca en qué chunk queda una frase importante de tu documento con cada estrategia.
¿Queda completa? ¿Queda con el contexto necesario para entenderla?

In [ ]:
FRASE = {"cafeteria": "La descalcificación se realiza cada 3 meses",
         "streamly": "expiran después de 3600 segundos",
         "veterinaria": "Usa el collar isabelino"}.get(DOCUMENTO, texto[1000:1040])

estrategias = {
    "caracteres 150/0": chunk_text(texto, 150, 0),
    "caracteres 600/100": chunk_text(texto, 600, 100),
    "párrafos 800": chunk_by_paragraph(texto, 800),
    "secciones": [c["texto"] for c in chunk_by_section(texto)],
}
for nombre, chunks in estrategias.items():
    encontrados = [c for c in chunks if FRASE in c]
    print("═" * 80)
    print(f"{nombre}: {len(chunks)} chunks | ¿frase completa en algún chunk? {'sí' if encontrados else 'NO — quedó partida'}")
    if encontrados:
        print(encontrados[0][:500])

## Pregunta de cierre

**¿Qué configuración preserva mejor el significado de tu documento? Justifica con lo que observaste.**

> ...